In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

spark.sql("USE workspace.colloky")
bronze = spark.table("colloky_bronze_snapshots")
print(f"Filas en Bronze: {bronze.count():,}")

Filas en Bronze: 53,469


In [0]:
TYPES = {
    "date_scraped": "timestamp",
    "page_number": "int",
    "page_position": "int",
    "category_rank": "int",
    "sku_number": "int",
    "age_min_months": "int",
    "color_count": "int",
    "original_price": "int",
    "final_price": "int",
    "discount": "decimal(6,4)",
    "stockout_rate": "decimal(6,4)",
    "offers_total": "int",
    "offers_in_stock": "int",
    "offers_out_of_stock": "int",
    "is_licensed": "boolean",
}

df = bronze

# Quitar espacios y convertir textos vacíos en nulos
for col_name, col_type in df.dtypes:
    if col_type == "string" and not col_name.startswith("_"):
        df = df.withColumn(
            col_name,
            F.when(F.trim(F.col(col_name)) == "", None).otherwise(F.trim(F.col(col_name)))
        )

# Convertir cada columna a su tipo (try_cast deja nulo si un valor no se puede convertir)
for col_name, col_type in TYPES.items():
    if col_name in df.columns:
        df = df.withColumn(col_name, F.expr(f"try_cast(`{col_name}` AS {col_type})"))

df = df.withColumn("snapshot_date", F.to_date("date_scraped"))
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- date_scraped: timestamp (nullable = true)
 |-- brand: string (nullable = true)
 |-- product_url: string (nullable = true)
 |-- category: string (nullable = true)
 |-- page_number: integer (nullable = true)
 |-- page_position: integer (nullable = true)
 |-- category_rank: integer (nullable = true)
 |-- sku: string (nullable = true)
 |-- sku_number: integer (nullable = true)
 |-- image_url: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_name_raw: string (nullable = true)
 |-- product_description: string (nullable = true)
 |-- description_main: string (nullable = true)
 |-- care_tips: string (nullable = true)
 |-- when_to_use: string (nullable = true)
 |-- pairs_with: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age_segment: string (nullable = true)
 |-- age_min_months: integer (nullable = true)
 |-- size_range: string (nullable = true)
 |-- is_

In [0]:
# Un registro por producto y corrida. Si hay duplicados, se queda el de menos stock
# (la lectura más conservadora, igual que la regla del scraper).
w = Window.partitionBy("product_url", "date_scraped").orderBy(F.col("offers_in_stock").asc_nulls_last())

before = df.count()
df = df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")
after = df.count()

print(f"Duplicados eliminados: {before - after:,}")

Duplicados eliminados: 0


In [0]:
COMPLETENESS_THRESHOLD = 0.90

runs = df.groupBy("date_scraped").agg(F.countDistinct("product_url").alias("products"))
median_products = runs.approxQuantile("products", [0.5], 0)[0]

runs = (
    runs
    .withColumn("expected_products", F.lit(int(median_products)))
    .withColumn("completeness", F.round(F.col("products") / F.col("expected_products"), 3))
    .withColumn("is_complete_run", F.col("products") >= F.lit(median_products * COMPLETENESS_THRESHOLD))
)

runs.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("colloky_silver_runs")

print(f"Corrida típica (mediana): {int(median_products):,} productos")
display(runs.orderBy("date_scraped"))

Corrida típica (mediana): 3,322 productos


date_scraped,products,expected_products,completeness,is_complete_run
2026-10-06T18:42:00.000Z,3379,3322,1.017,true
2026-10-06T19:35:00.000Z,3439,3322,1.035,true
2026-10-06T20:30:00.000Z,3383,3322,1.018,true
2026-10-06T21:25:00.000Z,3432,3322,1.033,true
2026-10-06T22:27:00.000Z,3438,3322,1.035,true
2026-10-06T23:51:00.000Z,3429,3322,1.032,true
2026-10-07T09:24:00.000Z,3204,3322,0.964,true
2026-10-07T15:06:00.000Z,3257,3322,0.98,true
2026-10-07T16:01:00.000Z,3266,3322,0.983,true
2026-10-07T17:05:00.000Z,3314,3322,0.998,true


In [0]:
df = df.join(runs.select("date_scraped", "is_complete_run"), on="date_scraped", how="left")

df = df.withColumn(
    "dq_issues",
    F.concat_ws(
        ", ",
        F.when(F.col("final_price").isNull() | (F.col("final_price") <= 0), F.lit("invalid_final_price")),
        F.when(F.col("final_price") > F.col("original_price"), F.lit("final_above_list_price")),
        F.when((F.col("discount") < 0) | (F.col("discount") > 1), F.lit("discount_out_of_range")),
        F.when((F.col("stockout_rate") < 0) | (F.col("stockout_rate") > 1), F.lit("stockout_out_of_range")),
        F.when(F.col("sku").isNull(), F.lit("missing_sku")),
    ),
)
df = df.withColumn("dq_issues", F.when(F.col("dq_issues") == "", None).otherwise(F.col("dq_issues")))

In [0]:
(
    df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("colloky_silver_snapshots")
)
print("✅ Tabla colloky_silver_snapshots creada")

✅ Tabla colloky_silver_snapshots creada


In [0]:
%sql
SELECT
    COALESCE(dq_issues, 'OK') AS quality_status,
    COUNT(*)                  AS rows
FROM colloky_silver_snapshots
GROUP BY 1
ORDER BY rows DESC

quality_status,rows
OK,53469
